# pMOT single-atom trajectory laboratory

This is the pMOT counterpart of the efficient multilevel MOT single-trajectory notebook. It uses the same repumper-included 24-state population-rate kernel, but removes the external magnetic field and obtains a local quantization-axis proxy from the vector AC Stark shifts of the six 1529-nm traveling components.

The notebook provides:

- full-sphere launch geometry, including a perpendicular-disc impact parameter;
- editable cooling, repump, trapping-beam, and integration parameters;
- independent propagation-frame polarization controls for all 18 traveling components;
- true-scale beam and trajectory rendering;
- position, velocity, force, scattering, effective-field, quantization-axis, and spherical-polarization diagnostics;
- beamwise effective-field and polarization-history CSV exports;
- optional comparison against a matched zero-trapping-power optical-molasses control.

Nothing is simulated merely by opening or running all cells. Choose a preset or edit the controls, then press **Run pMOT trajectory**.

In [ ]:
import os
import sys
import json
from dataclasses import asdict, replace
from datetime import datetime
from pathlib import Path
from time import perf_counter

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "pyproject.toml").is_file() and (PROJECT_ROOT / "src" / "pmot").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Could not locate the pMOT repository root.")

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

try:
    backend = "inline" if os.environ.get("PMOT_NOTEBOOK_HEADLESS") == "1" else "widget"
    get_ipython().run_line_magic("matplotlib", backend)
except (ImportError, ModuleNotFoundError):
    get_ipython().run_line_magic("matplotlib", "inline")
    backend = "inline"

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ModuleNotFoundError:
    widgets = None
    WIDGETS_AVAILABLE = False

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from pmot.mot_multilevel.rate_equations import RateEquationTrajectoryConfig
from pmot.pmot.trajectory_plotting import plot_pmot_trajectory_diagnostics
from pmot.pmot.vector_only_trajectories import (
    PMOTBeamHelicities,
    build_vector_only_apparatus,
    build_vector_only_trajectory_context,
    inward_launch_state,
    simulate_vector_only_pmot_trajectory,
    vector_only_trajectory_dataframe,
)
from pmot.pmot.vector_only_trajectory_diagnostics import (
    plot_vector_only_field_axis_polarization,
    vector_only_component_field_dataframe,
    vector_only_polarization_dataframe,
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "trajectories"
    / "pmot"
    / "vector_only_local_axis_notebook"
)
print(f"Project root: {PROJECT_ROOT}")
print(f"Matplotlib backend: {backend}")
print(f"Interactive widgets available: {WIDGETS_AVAILABLE}")
print(f"Run outputs: {OUTPUT_ROOT}")

## Physics used by this notebook

For trapping component \(j\), the atom-frame wavelength and stretched-transition-equivalent vector field are evaluated at every external timestep:

\[
\lambda'_j=\frac{\lambda_0}{1-\hat{\mathbf k}_j\cdot\mathbf v/c},
\qquad
\mathbf B_{\mathrm{eq},j}
=
\frac{
-\alpha_j^{(1)}(\lambda'_j)
[2I_j(\mathbf r)/(c\epsilon_0)]
\xi_j\hat{\mathbf k}_j
}{
\mu_{\mathrm{cycling}}
}.
\]

The component vectors are summed and normalized:

\[
\mathbf B_{\mathrm{eq}}=\sum_j\mathbf B_{\mathrm{eq},j},
\qquad
\hat{\mathbf q}=\mathbf B_{\mathrm{eq}}/|\mathbf B_{\mathrm{eq}}|.
\]

Every 780-nm cooling and repump component is decomposed into \(\sigma^+,\pi,\sigma^-\) relative to \(\hat{\mathbf q}\), after which the usual 24-state rate equation is solved. The effective detuning is

\[
\Delta_{\mathrm{eff},b,ge}
=
\Delta_{L,b}
-\delta_{\mathrm{HFS},ge}
-\mathbf k_b\cdot\mathbf v
-\Delta\omega^{(1)}_{\mathrm{AC},ge}.
\]

The 1529-nm Doppler shift changes the wavelength used for the polarizability lookup; it is not added as a second 780-nm Doppler term. The real external magnetic field is exactly zero.

**Important boundary:** this remains a provisional ideal-magic, vector-only calculation. Scalar/tensor cancellation is imposed, and the model omits a conservative 1529-nm gradient force, trapping-light scattering/heating/loss, coherent interference, measured Jones transformations, and physical nonadiabatic evolution where \(\mathbf B_{\mathrm{eq}}=0\). At a numerical field zero the previous axis is retained. The 38.294486 mW/path setting is only the current 20 G/cm stretched-reference gradient proxy, not an experimental power recommendation. An axial atom traverses the ideal 2.234-µm focus in much less than 1 µs, so ordinary interactive timesteps can classify the mechanical trajectory but do not resolve the focal peak.

In [ ]:
AXES = ("x", "y", "z")
KNOWN_WORKING_XYZ = ("sigma+", "sigma+", "sigma-")
DEFAULT_HELICITIES = PMOTBeamHelicities(
    cooling_incident_xyz=KNOWN_WORKING_XYZ,
    cooling_retro_xyz=KNOWN_WORKING_XYZ,
    repump_incident_xyz=KNOWN_WORKING_XYZ,
    repump_retro_xyz=KNOWN_WORKING_XYZ,
    trapping_incident_xyz=KNOWN_WORKING_XYZ,
    trapping_retro_xyz=KNOWN_WORKING_XYZ,
)


def beam_manifest(context):
    rows = []
    for beam in context.cooling_repump_beams:
        rows.append(
            {
                "family": beam.family,
                "axis": beam.axis_name,
                "path": beam.propagation_sense,
                "k_hat": tuple(round(value, 3) for value in beam.direction),
                "propagation-frame polarization": beam.circular_polarization,
                "wavelength [nm]": 1e9 * beam.wavelength_m,
                "modeled component power [mW]": 1e3 * beam.power_w,
                "1/e^2 waist diameter [mm]": 2e3 * beam.beam_radius_m,
                "waist coordinate [mm]": np.nan,
            }
        )
    for beam in context.trapping_beams:
        rows.append(
            {
                "family": "trapping",
                "axis": beam.axis_name,
                "path": beam.propagation_sense,
                "k_hat": tuple(round(value, 3) for value in beam.direction),
                "propagation-frame polarization": beam.helicity,
                "wavelength [nm]": 1e9 * beam.wavelength_m,
                "modeled component power [mW]": (
                    1e3
                    * context.trapping_power_w_per_path
                    * beam.power_per_incident_path_watt
                ),
                "1/e^2 waist diameter [mm]": 2e3 * beam.waist_radius_m,
                "waist coordinate [mm]": (
                    1e3 * np.dot(beam.waist_position_m, beam.direction)
                ),
            }
        )
    return pd.DataFrame(rows)


def _context_with_gravity(context, include_gravity):
    if context.multilevel_config.include_gravity == include_gravity:
        return context
    config = replace(
        context.multilevel_config,
        include_gravity=bool(include_gravity),
    )
    return replace(
        context,
        multilevel_config=config,
        _observable_context=replace(
            context._observable_context,
            multilevel_config=config,
        ),
    )


def _json_ready(value):
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return value.item()
    return value


def _linear_crossings(times, values, companion):
    times = np.asarray(times, dtype=float)
    values = np.asarray(values, dtype=float)
    companion = np.asarray(companion, dtype=float)
    rows = []
    for index in np.flatnonzero(values[:-1] * values[1:] < 0.0):
        fraction = -values[index] / (values[index + 1] - values[index])
        rows.append(
            {
                "time_ms": 1e3
                * float(
                    times[index]
                    + fraction * (times[index + 1] - times[index])
                ),
                "companion": float(
                    companion[index]
                    + fraction * (companion[index + 1] - companion[index])
                ),
                "index_before": int(index),
            }
        )
    return rows


def axial_crossing_summary(record):
    base = record.rate_equation
    time = np.asarray(base.times_s)
    position = np.asarray(base.positions_m)
    velocity = np.asarray(base.velocities_m_per_s)
    force = np.asarray(base.forces_n)
    x = position[:, 0]
    vx = velocity[:, 0]
    fx = force[:, 0]
    crossings = _linear_crossings(time, x, vx)
    turns = _linear_crossings(time, vx, x)
    formatted_crossings = [
        {
            "time_ms": item["time_ms"],
            "vx_m_per_s": item["companion"],
        }
        for item in crossings
    ]
    formatted_turns = []
    for item in turns:
        index = item["index_before"]
        fraction = -vx[index] / (vx[index + 1] - vx[index])
        interpolated_force = float(
            fx[index] + fraction * (fx[index + 1] - fx[index])
        )
        x_m = item["companion"]
        formatted_turns.append(
            {
                "time_ms": item["time_ms"],
                "x_mm": 1e3 * x_m,
                "force_x_n": interpolated_force,
                "force_times_displacement_j": interpolated_force * x_m,
                "restoring_at_turn": bool(interpolated_force * x_m < 0.0),
            }
        )
    return {
        "x_origin_crossing_count": len(formatted_crossings),
        "x_origin_crossings": formatted_crossings,
        "vx_turning_point_count": len(formatted_turns),
        "vx_turning_points": formatted_turns,
    }


def plot_matched_control_comparison(record, control, path=None):
    figure, panels = plt.subplots(2, 2, figsize=(13.0, 8.2))
    styles = (
        (record.rate_equation, "trapping shift on", "#2563eb"),
        (control.rate_equation, "zero trapping-power control", "#dc2626"),
    )
    for base, label, color in styles:
        time_ms = 1e3 * np.asarray(base.times_s)
        position = np.asarray(base.positions_m)
        velocity = np.asarray(base.velocities_m_per_s)
        radius_mm = 1e3 * np.linalg.norm(position, axis=1)
        speed = np.linalg.norm(velocity, axis=1)
        panels[0, 0].plot(time_ms, 1e3 * position[:, 0], color=color, label=label)
        panels[0, 1].plot(time_ms, velocity[:, 0], color=color, label=label)
        panels[1, 0].plot(time_ms, radius_mm, color=color, label=label)
        panels[1, 1].plot(1e3 * position[:, 0], velocity[:, 0], color=color, label=label)
    panels[0, 0].set(title="Axial position", ylabel="x [mm]")
    panels[0, 1].set(title="Axial velocity", ylabel="vx [m/s]")
    panels[1, 0].set(title="Distance from origin", ylabel="radius [mm]")
    panels[1, 1].set(title="Axial phase space", xlabel="x [mm]", ylabel="vx [m/s]")
    for panel in panels.flat:
        if panel is not panels[1, 1]:
            panel.set_xlabel("Time [ms]")
        panel.axhline(0.0, color="black", linewidth=0.7)
        panel.grid(alpha=0.22)
        panel.legend(frameon=False, fontsize=8)
    panels[1, 1].axvline(0.0, color="black", linewidth=0.7)
    figure.suptitle("Matched trapping-on and zero-power trajectory comparison")
    figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
    if path is not None:
        path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    return figure, panels


def run_pmot_case(
    *,
    radial_distance_mm=15.0,
    speed_m_per_s=17.0,
    polar_angle_deg=90.0,
    azimuth_angle_deg=0.0,
    impact_parameter_mm=0.0,
    impact_azimuth_deg=0.0,
    duration_ms=25.0,
    time_step_us=2.5,
    include_diffusion=False,
    include_gravity=True,
    seed=20260901,
    escape_radius_mm=30.0,
    cooling_power_mw=27.0,
    repump_power_mw=0.1,
    cooling_detuning_mhz=-15.0,
    repump_detuning_mhz=0.0,
    beam_diameter_mm=12.7,
    trapping_wavelength_nm=1529.268881,
    focus_offset_mm=10.0,
    input_beam_diameter_mm=35.0,
    focal_length_mm=80.3,
    retro_power_fraction=1.0,
    derive_trapping_power=True,
    target_gradient_g_per_cm=20.0,
    explicit_trapping_power_mw=38.294486,
    helicities=DEFAULT_HELICITIES,
    plot_extent_mm=30.0,
    run_zero_power_control=False,
    save_outputs=True,
    output_tag=None,
    progress_widget=None,
):
    apparatus = build_vector_only_apparatus(
        cooling_power_w_per_beam=1e-3 * cooling_power_mw,
        repump_power_w_per_beam=1e-3 * repump_power_mw,
        cooling_detuning_hz=1e6 * cooling_detuning_mhz,
        repump_detuning_hz=1e6 * repump_detuning_mhz,
        beam_diameter_m=1e-3 * beam_diameter_mm,
        trapping_wavelength_m=1e-9 * trapping_wavelength_nm,
        trapping_focus_offset_m=1e-3 * focus_offset_mm,
        trapping_input_beam_diameter_m=1e-3 * input_beam_diameter_mm,
        trapping_focal_length_m=1e-3 * focal_length_mm,
        trapping_retro_power_fraction=retro_power_fraction,
    )
    requested_power = (
        None if derive_trapping_power else 1e-3 * explicit_trapping_power_mw
    )
    context = build_vector_only_trajectory_context(
        apparatus=apparatus,
        helicities=helicities,
        trapping_power_w_per_path=requested_power,
        target_gradient_g_per_cm=target_gradient_g_per_cm,
    )
    context = _context_with_gravity(context, include_gravity)
    launch = inward_launch_state(
        radial_distance_m=1e-3 * radial_distance_mm,
        speed_m_per_s=speed_m_per_s,
        polar_angle_deg=polar_angle_deg,
        azimuth_angle_deg=azimuth_angle_deg,
        impact_parameter_m=1e-3 * impact_parameter_mm,
        impact_azimuth_deg=impact_azimuth_deg,
    )
    numerical = RateEquationTrajectoryConfig(
        time_step_s=1e-6 * time_step_us,
        include_diffusion=include_diffusion,
        seed=int(seed),
        escape_radius_m=1e-3 * escape_radius_mm,
    )
    step_count = int(np.ceil(1e3 * duration_ms / time_step_us))
    run_count = 2 if run_zero_power_control else 1
    print(
        f"Running {run_count} trajectory calculation(s): "
        f"T={duration_ms:g} ms, dt={time_step_us:g} us, "
        f"{step_count:,} steps/run, v0={speed_m_per_s:g} m/s",
        flush=True,
    )
    print(
        f"External B=(0,0,0); cooling={cooling_power_mw:g} mW/component; "
        f"repump={repump_power_mw:g} mW/component; "
        f"trapping={1e3*context.trapping_power_w_per_path:.9g} mW/path "
        f"({context.trapping_power_source})",
        flush=True,
    )
    if time_step_us > 0.1:
        print(
            "CAUTION: this timestep does not resolve a direct traversal of the "
            "ideal 2.234-um trapping focus; treat focal maxima as unresolved.",
            flush=True,
        )

    def make_progress(label, offset, span):
        def progress(completed, total, time_s):
            fraction = completed / total
            if progress_widget is not None:
                progress_widget.value = int(round(offset + span * fraction))
                progress_widget.description = f"{100*fraction:4.0f}%"
            print(
                f"  {label}: {100*fraction:5.1f}% "
                f"(t={1e3*time_s:7.3f} ms)",
                flush=True,
            )
        return progress

    started = perf_counter()
    record = simulate_vector_only_pmot_trajectory(
        launch,
        duration_s=1e-3 * duration_ms,
        context=context,
        trajectory_config=numerical,
        progress_callback=make_progress(
            "trapping on",
            0,
            50 if run_zero_power_control else 100,
        ),
    )
    control = None
    control_context = None
    if run_zero_power_control:
        control_context = build_vector_only_trajectory_context(
            apparatus=apparatus,
            helicities=helicities,
            trapping_power_w_per_path=0.0,
        )
        control_context = _context_with_gravity(
            control_context,
            include_gravity,
        )
        control = simulate_vector_only_pmot_trajectory(
            launch,
            duration_s=1e-3 * duration_ms,
            context=control_context,
            trajectory_config=numerical,
            progress_callback=make_progress("zero-power control", 50, 50),
        )
    wall_time_s = perf_counter() - started

    frame = vector_only_trajectory_dataframe(record)
    component_field_frame = vector_only_component_field_dataframe(record, context)
    polarization_frame = vector_only_polarization_dataframe(record, context)
    axis_history = np.asarray(record.quantization_axes)
    field_history = np.asarray(record.effective_fields_t)
    axis_step_degrees = np.degrees(
        np.arccos(
            np.clip(
                np.sum(axis_history[1:] * axis_history[:-1], axis=1),
                -1.0,
                1.0,
            )
        )
    )
    crossing = axial_crossing_summary(record)
    capture = record.capture

    stamp = output_tag or datetime.now().strftime("interactive_%Y%m%d_%H%M%S")
    output_directory = OUTPUT_ROOT / stamp
    trajectory_path = output_directory / "trajectory.csv"
    component_field_path = output_directory / "beamwise_effective_fields.csv"
    polarization_path = output_directory / "local_spherical_polarization_weights.csv"
    metadata_path = output_directory / "run_metadata.json"
    trajectory_figure_path = output_directory / "trajectory_and_beams.png"
    axis_figure_path = output_directory / "effective_field_axis_polarization.png"
    control_csv_path = output_directory / "zero_power_control_trajectory.csv"
    control_figure_path = output_directory / "zero_power_control_comparison.png"

    trajectory_figure, trajectory_axes = plot_pmot_trajectory_diagnostics(
        record,
        list(context.cooling_repump_beams),
        list(context.trapping_beams),
        path=trajectory_figure_path if save_outputs else None,
        title=f"Vector-only local-axis pMOT: {speed_m_per_s:g} m/s launch",
        axial_extent_m=1e-3 * plot_extent_mm,
    )
    axis_figure, axis_axes = plot_vector_only_field_axis_polarization(
        record,
        context,
        path=axis_figure_path if save_outputs else None,
        title=(
            "Beamwise-summed effective field, local quantization axis, "
            "and cooling polarization"
        ),
    )
    control_figure = None
    if control is not None:
        control_figure, _ = plot_matched_control_comparison(
            record,
            control,
            path=control_figure_path if save_outputs else None,
        )

    position = np.asarray(record.rate_equation.positions_m)
    velocity = np.asarray(record.rate_equation.velocities_m_per_s)
    force = np.asarray(record.rate_equation.forces_n)
    external_field = np.asarray(record.rate_equation.magnetic_fields_t)
    summary = pd.Series(
        {
            "termination": record.rate_equation.termination_reason,
            "capture classification": capture.classification,
            "trapped by current criterion": capture.trapped,
            "core entries": capture.core_entry_count,
            "continuous core residence [ms]": (
                1e3 * capture.maximum_continuous_core_residence_s
            ),
            "minimum radius [um]": 1e6 * capture.minimum_radius_m,
            "final radius [um]": 1e6 * capture.final_radius_m,
            "final speed [m/s]": np.linalg.norm(velocity[-1]),
            "x-origin crossings": crossing["x_origin_crossing_count"],
            "x-velocity turning points": crossing["vx_turning_point_count"],
            "peak optical force [zN]": (
                1e21 * np.max(np.linalg.norm(force, axis=1))
            ),
            "minimum effective-field proxy [G]": (
                1e4 * np.min(np.linalg.norm(field_history, axis=1))
            ),
            "maximum sampled effective-field proxy [G]": (
                1e4 * np.max(np.linalg.norm(field_history, axis=1))
            ),
            "maximum interstep axis rotation [deg]": (
                0.0 if not len(axis_step_degrees) else np.max(axis_step_degrees)
            ),
            "external magnetic field identically zero": bool(
                np.all(external_field == 0.0)
            ),
            "trapping power per incident path [mW]": (
                1e3 * context.trapping_power_w_per_path
            ),
            "stored samples": len(frame),
            "wall time including optional control [s]": wall_time_s,
        },
        name="value",
    )

    metadata = {
        "status": "PROVISIONAL_VECTOR_ONLY_LOCAL_AXIS_TRAJECTORY",
        "model": record.model_metadata,
        "helicities_propagation_frame": asdict(helicities),
        "launch": {
            "position_m": launch.position_m,
            "velocity_m_per_s": launch.velocity_m_per_s,
            "radial_distance_mm": radial_distance_mm,
            "speed_m_per_s": speed_m_per_s,
            "polar_angle_deg": polar_angle_deg,
            "azimuth_angle_deg": azimuth_angle_deg,
            "impact_parameter_mm": impact_parameter_mm,
            "impact_azimuth_deg": impact_azimuth_deg,
        },
        "integration": {
            "duration_ms": duration_ms,
            "time_step_us": time_step_us,
            "include_diffusion": include_diffusion,
            "include_gravity": include_gravity,
            "seed": int(seed),
            "escape_radius_mm": escape_radius_mm,
        },
        "optics": {
            "cooling_power_mw_per_component": cooling_power_mw,
            "repump_power_mw_per_component": repump_power_mw,
            "cooling_detuning_mhz": cooling_detuning_mhz,
            "repump_detuning_mhz": repump_detuning_mhz,
            "beam_diameter_mm": beam_diameter_mm,
            "trapping_wavelength_nm": trapping_wavelength_nm,
            "focus_offset_mm": focus_offset_mm,
            "input_beam_diameter_mm": input_beam_diameter_mm,
            "focal_length_mm": focal_length_mm,
            "retro_power_fraction": retro_power_fraction,
            "trapping_power_w_per_path": context.trapping_power_w_per_path,
            "trapping_power_source": context.trapping_power_source,
            "target_gradient_g_per_cm": context.target_gradient_g_per_cm,
        },
        "capture": asdict(capture),
        "axial_crossing_and_turning": crossing,
        "summary": summary.to_dict(),
        "matched_zero_power_control": (
            None
            if control is None
            else {
                "capture": asdict(control.capture),
                "axial_crossing_and_turning": axial_crossing_summary(control),
            }
        ),
    }
    if save_outputs:
        output_directory.mkdir(parents=True, exist_ok=True)
        frame.to_csv(trajectory_path, index=False)
        component_field_frame.to_csv(component_field_path, index=False)
        polarization_frame.to_csv(polarization_path, index=False)
        if control is not None:
            vector_only_trajectory_dataframe(control).to_csv(
                control_csv_path,
                index=False,
            )
        metadata["outputs"] = {
            "trajectory_csv": str(trajectory_path),
            "beamwise_field_csv": str(component_field_path),
            "polarization_csv": str(polarization_path),
            "trajectory_figure": str(trajectory_figure_path),
            "axis_figure": str(axis_figure_path),
            "control_csv": str(control_csv_path) if control is not None else None,
            "control_figure": (
                str(control_figure_path) if control is not None else None
            ),
        }
        metadata_path.write_text(
            json.dumps(_json_ready(metadata), indent=2) + "\n",
            encoding="utf-8",
        )

    plt.show()
    display(summary.to_frame())
    if crossing["x_origin_crossings"]:
        display(
            pd.DataFrame(crossing["x_origin_crossings"]).rename(
                columns={"time_ms": "crossing time [ms]", "vx_m_per_s": "vx [m/s]"}
            )
        )
    if crossing["vx_turning_points"]:
        display(
            pd.DataFrame(crossing["vx_turning_points"]).rename(
                columns={"time_ms": "turn time [ms]", "x_mm": "x [mm]"}
            )
        )
    display(
        beam_manifest(context).style.format(
            {
                "wavelength [nm]": "{:.6f}",
                "modeled component power [mW]": "{:.6f}",
                "1/e^2 waist diameter [mm]": "{:.6f}",
                "waist coordinate [mm]": "{:.3f}",
            }
        )
    )
    if save_outputs:
        print(f"Saved run directory: {output_directory}")

    return {
        "record": record,
        "control_record": control,
        "context": context,
        "control_context": control_context,
        "frame": frame,
        "component_field_frame": component_field_frame,
        "polarization_frame": polarization_frame,
        "summary": summary,
        "crossing": crossing,
        "trajectory_figure": trajectory_figure,
        "axis_figure": axis_figure,
        "control_figure": control_figure,
        "output_directory": output_directory if save_outputs else None,
    }

## Interactive trajectory controls

The launch direction spans the full sphere: polar angle \(0^\circ\) to \(180^\circ\) and azimuth \(0^\circ\) to \(360^\circ\). A nonzero impact parameter offsets the launch point within the plane perpendicular to the incoming direction; the velocity remains parallel to the disc normal.

The default propagation-frame helicities are the currently identified damping/restoring tuple \((\sigma^+,\sigma^+,\sigma^-)\) on \(x,y,z\) for both incident and retro components. A trapping component presently accepts only \(\sigma^\pm\), because the provisional tensor machinery does not define the required transverse axis for a \(\pi\)-polarized trapping beam.

Select **matched zero-power control** when you want to distinguish pMOT displacement restoration from ordinary 780-nm velocity damping. This doubles the trajectory runtime.

In [ ]:
def build_interactive_controls():
    if not WIDGETS_AVAILABLE:
        display(
            Markdown(
                "**ipywidgets is not installed in this kernel.** Install the "
                "project notebook extras, or use the programmatic examples "
                "in the next cell."
            )
        )
        return None

    def number(description, value, *, step=0.1, minimum=None, maximum=None):
        kwargs = {
            "description": description,
            "value": value,
            "step": step,
            "continuous_update": False,
        }
        if minimum is not None:
            kwargs["min"] = minimum
        if maximum is not None:
            kwargs["max"] = maximum
        widget_class = (
            widgets.BoundedFloatText
            if minimum is not None and maximum is not None
            else widgets.FloatText
        )
        return widget_class(**kwargs)

    preset = widgets.Dropdown(
        description="preset",
        options=(
            "standard axial 17 m/s",
            "axial restoration 25 m/s",
            "oblique (2,3,6)/7 at 17 m/s",
            "custom",
        ),
        value="standard axial 17 m/s",
    )
    launch = {
        "radial_distance_mm": number(
            "distance [mm]", 15.0, step=0.5, minimum=0.1, maximum=100.0
        ),
        "speed_m_per_s": number(
            "speed [m/s]", 17.0, step=0.25, minimum=0.0, maximum=100.0
        ),
        "polar_angle_deg": number(
            "polar [deg]", 90.0, step=1.0, minimum=0.0, maximum=180.0
        ),
        "azimuth_angle_deg": number(
            "azimuth [deg]", 0.0, step=1.0, minimum=0.0, maximum=360.0
        ),
        "impact_parameter_mm": number(
            "impact [mm]", 0.0, step=0.25, minimum=0.0, maximum=50.0
        ),
        "impact_azimuth_deg": number(
            "impact angle", 0.0, step=1.0, minimum=0.0, maximum=360.0
        ),
    }
    integration = {
        "duration_ms": number(
            "duration [ms]", 25.0, step=1.0, minimum=0.01, maximum=500.0
        ),
        "time_step_us": number(
            "dt [us]", 2.5, step=0.25, minimum=0.01, maximum=100.0
        ),
        "escape_radius_mm": number(
            "escape [mm]", 30.0, step=1.0, minimum=1.0, maximum=500.0
        ),
        "seed": widgets.BoundedIntText(
            description="seed",
            value=20260901,
            min=0,
            max=2_147_483_647,
        ),
        "include_gravity": widgets.Checkbox(
            description="gravity",
            value=True,
            indent=False,
        ),
        "include_diffusion": widgets.Checkbox(
            description="recoil diffusion",
            value=False,
            indent=False,
        ),
    }
    mot_light = {
        "cooling_power_mw": number(
            "cooling [mW]", 27.0, step=0.5, minimum=0.0, maximum=1000.0
        ),
        "repump_power_mw": number(
            "repump [mW]", 0.1, step=0.01, minimum=0.0, maximum=100.0
        ),
        "cooling_detuning_mhz": number("cool det. [MHz]", -15.0, step=0.5),
        "repump_detuning_mhz": number("rep det. [MHz]", 0.0, step=0.1),
        "beam_diameter_mm": number(
            "780 diam. [mm]", 12.7, step=0.1, minimum=0.1, maximum=100.0
        ),
    }
    trapping = {
        "trapping_wavelength_nm": number(
            "lambda [nm]",
            1529.268881,
            step=0.000001,
            minimum=1500.0,
            maximum=1600.0,
        ),
        "focus_offset_mm": number(
            "focus offset", 10.0, step=0.5, minimum=0.01, maximum=100.0
        ),
        "input_beam_diameter_mm": number(
            "input diam.", 35.0, step=0.5, minimum=0.1, maximum=200.0
        ),
        "focal_length_mm": number(
            "focal len.", 80.3, step=0.1, minimum=0.1, maximum=1000.0
        ),
        "retro_power_fraction": number(
            "retro fraction", 1.0, step=0.05, minimum=0.0, maximum=10.0
        ),
        "target_gradient_g_per_cm": number(
            "gradient proxy", 20.0, step=1.0, minimum=0.01, maximum=1000.0
        ),
        "explicit_trapping_power_mw": number(
            "trap power [mW]",
            38.294486,
            step=0.1,
            minimum=0.0,
            maximum=10000.0,
        ),
        "derive_trapping_power": widgets.Checkbox(
            description="derive power from gradient proxy",
            value=True,
            indent=False,
        ),
        "plot_extent_mm": number(
            "plot extent", 30.0, step=1.0, minimum=1.0, maximum=200.0
        ),
    }

    helicity_widgets = {}
    grid_items = [
        widgets.HTML("<b>family / path</b>"),
        widgets.HTML("<b>x</b>"),
        widgets.HTML("<b>y</b>"),
        widgets.HTML("<b>z</b>"),
    ]
    for family in ("cooling", "repump", "trapping"):
        for sense in ("incident", "retro"):
            grid_items.append(widgets.HTML(f"<b>{family} {sense}</b>"))
            for index, axis_name in enumerate(AXES):
                options = (
                    ("sigma+", "sigma-")
                    if family == "trapping"
                    else ("sigma+", "sigma-", "pi")
                )
                dropdown = widgets.Dropdown(
                    options=options,
                    value=KNOWN_WORKING_XYZ[index],
                    description="",
                    layout=widgets.Layout(width="125px"),
                )
                helicity_widgets[(family, sense, axis_name)] = dropdown
                grid_items.append(dropdown)
    helicity_grid = widgets.GridBox(
        grid_items,
        layout=widgets.Layout(
            grid_template_columns="170px 130px 130px 130px",
            grid_gap="4px 8px",
            align_items="center",
        ),
    )

    save_checkbox = widgets.Checkbox(
        description="save CSV, JSON, and PNG",
        value=True,
        indent=False,
    )
    control_checkbox = widgets.Checkbox(
        description="matched zero-power control",
        value=False,
        indent=False,
    )
    run_button = widgets.Button(
        description="Run pMOT trajectory",
        button_style="primary",
        icon="play",
    )
    progress_bar = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description="ready",
    )
    run_output = widgets.Output()

    def apply_preset(change):
        selected = change["new"]
        if selected == "custom":
            return
        launch["radial_distance_mm"].value = 15.0
        launch["impact_parameter_mm"].value = 0.0
        launch["impact_azimuth_deg"].value = 0.0
        if selected == "standard axial 17 m/s":
            launch["speed_m_per_s"].value = 17.0
            launch["polar_angle_deg"].value = 90.0
            launch["azimuth_angle_deg"].value = 0.0
        elif selected == "axial restoration 25 m/s":
            launch["speed_m_per_s"].value = 25.0
            launch["polar_angle_deg"].value = 90.0
            launch["azimuth_angle_deg"].value = 0.0
            control_checkbox.value = True
            integration["duration_ms"].value = 10.0
            integration["time_step_us"].value = 1.25
        else:
            launch["speed_m_per_s"].value = 17.0
            launch["polar_angle_deg"].value = np.degrees(np.arccos(6.0 / 7.0))
            launch["azimuth_angle_deg"].value = np.degrees(np.arctan2(3.0, 2.0))

    preset.observe(apply_preset, names="value")

    def selected_helicities():
        values = {}
        for family in ("cooling", "repump", "trapping"):
            for sense in ("incident", "retro"):
                values[f"{family}_{sense}_xyz"] = tuple(
                    helicity_widgets[(family, sense, axis_name)].value
                    for axis_name in AXES
                )
        return PMOTBeamHelicities(**values)

    def values(group):
        return {name: widget.value for name, widget in group.items()}

    def rows(title, controls):
        items = list(controls.values())
        return widgets.VBox(
            [widgets.HTML(f"<b>{title}</b>")]
            + [
                widgets.HBox(items[index : index + 3])
                for index in range(0, len(items), 3)
            ]
        )

    def run_from_widgets(_=None):
        run_button.disabled = True
        progress_bar.value = 0
        progress_bar.description = "0%"
        progress_bar.bar_style = ""
        with run_output:
            run_output.clear_output(wait=True)
            plt.close("all")
            try:
                result = run_pmot_case(
                    **values(launch),
                    **values(integration),
                    **values(mot_light),
                    **values(trapping),
                    helicities=selected_helicities(),
                    run_zero_power_control=control_checkbox.value,
                    save_outputs=save_checkbox.value,
                    progress_widget=progress_bar,
                )
                globals()["PMOT_RUN"] = result
                progress_bar.value = 100
                progress_bar.description = "done"
            except Exception:
                progress_bar.bar_style = "danger"
                progress_bar.description = "error"
                raise
            else:
                progress_bar.bar_style = "success"
            finally:
                run_button.disabled = False

    run_button.on_click(run_from_widgets)
    panel = widgets.VBox(
        [
            widgets.HTML("<b>Preset</b>"),
            preset,
            rows("Launch geometry", launch),
            rows("Integration", integration),
            rows("Cooling and repump", mot_light),
            rows("1529-nm geometry and power", trapping),
            widgets.HTML(
                "<b>Propagation-frame polarization of every traveling component</b>"
            ),
            helicity_grid,
            widgets.HBox(
                [
                    save_checkbox,
                    control_checkbox,
                    run_button,
                    progress_bar,
                ]
            ),
            run_output,
        ]
    )
    display(panel)
    return {
        "panel": panel,
        "preset": preset,
        "launch": launch,
        "integration": integration,
        "mot_light": mot_light,
        "trapping": trapping,
        "helicity": helicity_widgets,
    }


PMOT_CONTROLS = build_interactive_controls()

## Programmatic presets

The interactive panel is the normal entry point. The dictionaries below provide equivalent script-like calls and remain useful in kernels without ipywidgets. Uncomment one call to execute it.

In [ ]:
STANDARD_AXIAL_17MPS = dict(
    radial_distance_mm=15.0,
    speed_m_per_s=17.0,
    polar_angle_deg=90.0,
    azimuth_angle_deg=0.0,
    impact_parameter_mm=0.0,
    duration_ms=25.0,
    time_step_us=2.5,
)

AXIAL_RESTORATION_25MPS = dict(
    radial_distance_mm=15.0,
    speed_m_per_s=25.0,
    polar_angle_deg=90.0,
    azimuth_angle_deg=0.0,
    impact_parameter_mm=0.0,
    duration_ms=10.0,
    time_step_us=1.25,
    run_zero_power_control=True,
)

OBLIQUE_236_17MPS = dict(
    radial_distance_mm=15.0,
    speed_m_per_s=17.0,
    polar_angle_deg=np.degrees(np.arccos(6.0 / 7.0)),
    azimuth_angle_deg=np.degrees(np.arctan2(3.0, 2.0)),
    impact_parameter_mm=0.0,
    duration_ms=25.0,
    time_step_us=2.5,
)

# PMOT_RUN = run_pmot_case(**STANDARD_AXIAL_17MPS)
# PMOT_RUN = run_pmot_case(**AXIAL_RESTORATION_25MPS)
# PMOT_RUN = run_pmot_case(**OBLIQUE_236_17MPS)

## Reading the results

A trajectory is marked trapped when it either remains continuously inside the central 2-mm core for at least 5 ms or enters that core twice with an intervening exit. This is an inexpensive trajectory classifier—not proof of a dynamically stable pMOT.

For an axial restoration test, look for all three of the following:

1. the atom crosses \(x=0\);
2. it reaches a turning point on the opposite side with \(v_x=0\);
3. at that turn, \(F_xx<0\), so the force points back toward the origin.

The local-axis figure shows the components and magnitude of the summed transition-equivalent field, the normalized quantization-axis history, and the time-averaged cooling-beam spherical components. The exported polarization CSV retains the full \(\sigma^+,\pi,\sigma^-\) decomposition of every cooling, repump, and trapping component at every stored timestep.

One-dimensional restoration does not establish three-dimensional stability. Before interpreting capture quantitatively, the model still needs a full state-resolved Stark Hamiltonian, realistic trapping-beam parameters, focal timestep resolution, conservative Stark force, 1529-nm scattering/heating, polarization transformations, and a physical treatment of the fictitious-field zero.